# Decision-time feature policy and label-maturity audit

**Research question:** Which application-time fields are eligible without leaking cohort, identity, or post-decision outcomes?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Absolute dates should become recency relative to date_decision.
- Eligibility rules must be fitted on development only.
- Late-cohort prevalence decline may reflect outcome maturity, not only drift.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: credit-risk-decision-lab


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,numeric_static,application static,train_static_0_*.parquet,application,Numeric fields eligible under frozen manifest,at date_decision,yes,development,native missing for XGBoost; median/binning for...,model,NaN,medium: definitions reviewed for post-decision...,credit-history variables may create access pro...,"feature inventory, time stability, baseline co..."
1,categorical_static,application static,train_static_0_* and train_static_cb_0,application,Low-cardinality categories eligible under froz...,at date_decision,yes,development,unknown/encoded from development,model,NaN,medium,category values may proxy protected characteri...,cardinality and missingness gates
2,relative_date_D,date recency,source columns ending D,application,date_decision - source date in days,at date_decision,yes,development,native missing,model,NaN,low after relative conversion,history length can proxy age/tenure,post-decision date check
3,missing_indicator,coverage,selected source fields,application,1 when source value is missing,at date_decision,yes,development,explicit binary,linear baseline,NaN,medium: provider/product eligibility signal,coverage can proxy channel or geography,cohort missingness monitoring
4,quantile_bin,nonlinear baseline,eligible numeric features,application,Ten development-fitted quantile bins with one-...,at date_decision,yes,development,imputer inside pipeline,binned-logit baseline,NaN,low,inherits source feature risks,locked OOT baseline comparison
5,case_id,identifier,train_base.case_id,application,No transformation,at decision,no model use,none,not applicable,excluded,identifier,high: identifier leakage,identity,excluded by contract
6,WEEK_NUM/MONTH/date_decision,partition/time,train_base,application,Used only for chronological partitions,at decision,audit only,none,not applicable,excluded,partition leakage,high: direct cohort leakage,time may proxy policy changes,excluded by contract
7,sensitive-token fields,protected/proxy,source fields matching birth/marital/education...,application,No transformation,at decision,excluded,development,not applicable,excluded,sensitive/proxy governance,high governance risk,protected or close proxy,frozen exclusion list


,feature_family,model_role,features
0,application static,model,2
1,coverage,linear baseline,1
2,date recency,model,1
3,identifier,excluded,1
4,nonlinear baseline,binned-logit baseline,1
5,partition/time,excluded,1
6,protected/proxy,excluded,1


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,numeric_static,at date_decision,yes,development,medium: definitions reviewed for post-decision...,"feature inventory, time stability, baseline co..."
1,categorical_static,at date_decision,yes,development,medium,cardinality and missingness gates
2,relative_date_D,at date_decision,yes,development,low after relative conversion,post-decision date check
3,missing_indicator,at date_decision,yes,development,medium: provider/product eligibility signal,cohort missingness monitoring
4,quantile_bin,at date_decision,yes,development,low,locked OOT baseline comparison
5,case_id,at decision,no model use,none,high: identifier leakage,excluded by contract
6,WEEK_NUM/MONTH/date_decision,at decision,audit only,none,high: direct cohort leakage,excluded by contract
7,sensitive-token fields,at decision,excluded,development,high governance risk,frozen exclusion list


## Committed analytical evidence

In [4]:
evidence_files = ["reports/feature_inventory.csv", "reports/label_maturity_diagnostic.json"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/feature_inventory.csv`

,feature,description,source_table,transformed_type,development_missing_rate,oot_missing_rate,status,reason,date_treatment
0,clientscnt_136L,Number of applications associated with same em...,train_static_0,Float64,0.999644,1.000000,excluded,More than 98% missing in development,not applicable
1,deferredmnthsnum_166L,Number of deferred months.,train_static_0,Float64,0.000000,0.000000,excluded,Constant or unsupported type in development,not applicable
2,interestrategrace_34L,Interest rate during the grace period.,train_static_0,Float64,0.984581,0.997218,excluded,More than 98% missing in development,not applicable
3,isbidproductrequest_292L,Flag indicating if the product is a cross-sell.,train_static_0,Boolean,0.991553,0.992644,excluded,More than 98% missing in development,not applicable
4,lastapprcommoditytypec_5251766M,Commodity type of the last application.,train_static_0,String,0.000000,0.000000,excluded,Constant or unsupported type in development,not applicable
5,lastotherinc_902A,Amount of other income reported by the client ...,train_static_0,Float64,0.997850,0.998372,excluded,More than 98% missing in development,not applicable
6,lastotherlnsexpense_631A,Monthly expenses on other loans from the last ...,train_static_0,Float64,0.997886,0.998266,excluded,Sensitive/protected or proxy-risk policy,not applicable
7,lastrejectcommodtypec_5251769M,Commodity type of the last rejected application.,train_static_0,String,0.000000,0.000000,excluded,Constant or unsupported type in development,not applicable
8,lastrepayingdate_696D,Date of the last payment made by the applicant.,train_static_0,Float32,0.998365,0.999354,excluded,More than 98% missing in development,days before decision date
9,payvacationpostpone_4187118D,Date of last payment holiday instalment.,train_static_0,Float32,0.998351,0.958374,excluded,More than 98% missing in development,days before decision date


### `reports/label_maturity_diagnostic.json`

{'reference_last_week': 54,
 'late_first_week': 78,
 'reference_default_rate': 0.031254149074590405,
 'late_default_rate': 0.021192166478804536,
 'relative_default_rate_change': -0.3219406988740017,
 'reference_mean_weekly_applications': 20541.272727272728,
 'late_mean_weekly_applications': 10836.214285714286,
 'relative_volume_change': -0.4724662668381301,
 'right_censoring_can_be_ruled_out': False,
 'reason': 'The public target has no documented outcome-window end date or per-row label-maturity timestamp. Cohort drift and right-censoring cannot be separated from this table alone.'}

## Decision

**The frozen feature policy supports research benchmarking, while maturity ambiguity blocks unqualified PD interpretation.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.